# Fine-Tuning Models in Azure AI Foundry

This notebook follows the same structure used in previous assignments: setup, guided tasks, implementation evidence, and analysis.

## Objective
Train and evaluate a fine-tuned model, compare it against a base model, and document findings clearly.

## 0) Setup and Configuration

Run this section first. It loads credentials and prepares reusable helpers for testing.

In [ ]:
# ============================================================================
# SETUP & CONFIGURATION: Load Azure credentials and initialize OpenAI client
# ============================================================================

import json
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve Azure credentials and deployment names from environment
AZURE_API_KEY = os.getenv("AZURE_API_KEY", "").strip()
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "").strip()
BASE_DEPLOYMENT_NAME = os.getenv("BASE_DEPLOYMENT_NAME", "").strip()
FINETUNED_DEPLOYMENT_NAME = os.getenv("FINETUNED_DEPLOYMENT_NAME", "").strip()

# Validate that critical credentials are present
required = {
    "AZURE_API_KEY": AZURE_API_KEY,
    "AZURE_ENDPOINT": AZURE_ENDPOINT,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}")

# Initialize OpenAI client pointing to Azure endpoint
client = OpenAI(base_url=AZURE_ENDPOINT, api_key=AZURE_API_KEY)

# Confirm successful setup and show deployment configuration status
print("Environment loaded successfully.")
print(f"Endpoint: {AZURE_ENDPOINT}")
print(f"Base deployment set: {bool(BASE_DEPLOYMENT_NAME)}")
print(f"Fine-tuned deployment set: {bool(FINETUNED_DEPLOYMENT_NAME)}")

Environment loaded successfully.
Endpoint: https://03-fine-tuning-resource.openai.azure.com/openai/v1
Base deployment set: True
Fine-tuned deployment set: True


---

## 1) Introduction and Dataset Context

Chosen use case (from task examples): technical support chatbot specialized in a specific topic.

In this assignment, the model is fine-tuned as **AzureFoundryCoach**, a beginner-friendly technical support assistant focused on Azure AI Foundry workflows.

- Problem it solves: students need clear, fast, and reliable step-by-step help to complete Foundry fine-tuning tasks.
- Dataset objective: teach the model to answer with practical steps, quick verification checks, and common pitfalls to avoid.
- Scope and constraints: concise guidance for Foundry setup, fine-tuning, deployment, and evaluation; avoid invented account-specific details and ask users to verify portal-specific values when needed.

### 1.1 Dataset Design Notes

- Domain: Azure AI Foundry technical support.
- Target users: beginners/students completing AI engineering assignments.
- Desired assistant tone/style: clear, supportive, concise, and procedural (numbered steps).
- Minimum examples prepared: 100 total examples (80 training, 20 validation).
- Why this dataset is suitable for fine-tuning: examples are consistent in structure, focus on practical Foundry tasks, include verification guidance, and reinforce the same response style needed for evaluation.

In [ ]:
# ============================================================================
# JSONL VALIDATION: Verify training and validation datasets are properly formatted
# ============================================================================

def validate_jsonl(file_path: Path):
    """
    Validate JSONL file format and structure for fine-tuning datasets.

    Expected format per line:
    {
        "messages": [
            {"role": "system", "content": "..."},
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]
    }

    Args:
        file_path: Path to JSONL file

    Returns:
        (lines_ok, errors): Tuple of valid line count and list of error messages
    """
    errors = []
    lines_ok = 0

    with file_path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            # Skip empty lines
            if not line:
                errors.append(f"Line {idx}: empty line")
                continue

            # Validate JSON syntax
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append(f"Line {idx}: invalid JSON ({exc})")
                continue

            # Verify 'messages' list exists and is non-empty
            messages = obj.get('messages')
            if not isinstance(messages, list) or not messages:
                errors.append(f"Line {idx}: missing messages list")
                continue

            # Ensure all required roles are present (system, user, assistant)
            roles = [m.get('role') for m in messages if isinstance(m, dict)]
            if 'system' not in roles or 'user' not in roles or 'assistant' not in roles:
                errors.append(
                    f"Line {idx}: must include system, user, assistant roles")
                continue

            # Count valid lines
            lines_ok += 1

    return lines_ok, errors


# Validate both training and validation datasets
train_path = Path('data/training_set.jsonl')
val_path = Path('data/validation_set.jsonl')

for path in [train_path, val_path]:
    if not path.exists():
        print(f"{path.name} not found in current folder.")
        continue

    # Run validation and display results
    ok, errs = validate_jsonl(path)
    print(f"\n{path.name} -> valid lines: {ok}")
    if errs:
        print("Errors:")
        # Show first 10 errors, summarize if more exist
        for e in errs[:10]:
            print(" -", e)
        if len(errs) > 10:
            print(f"... and {len(errs)-10} more")
    else:
        print("No format errors found.")


training_set.jsonl -> valid lines: 80
No format errors found.

validation_set.jsonl -> valid lines: 20
No format errors found.


---

## 2) Fine-Tuning Process (Portal Mode)

Use Azure AI Foundry Studio and document decisions.

Checklist:
1. Base model selected
2. Training type selected and justified
3. Hyperparameters (auto or manual)
4. Suffix chosen
5. Files uploaded
6. Job launched

### 2.1 Evidence

Portal recording for the fine-tuning job configuration:
- [part2_fine_tuning_portal_configuration.mp4](assets/part2_fine_tuning_portal_configuration.mp4)

<video src="assets/part2_fine_tuning_portal_configuration.mp4" controls width="900"></video>

What the video shows:
- Navigation to Azure AI Foundry Fine-tuning section
- Base model selection
- Training type and default hyperparameter configuration
- Training and validation file upload
- Fine-tuning job launch

Training type justification (Standard):
- Although the video shows the **Developer** selection step, the final submitted run used **Standard** training type.
- We selected **Standard** to prioritize training robustness and more stable optimization behavior for the final comparison stage.
- Standard training type is better aligned with obtaining stronger quality signals (loss trends and output consistency) for the assignment's base-vs-fine-tuned evaluation.
- It is a better fit for producing a reliable final model artifact intended for reportable results and qualitative analysis.

---

## 3) Metric Analysis

Load Foundry training metrics from `results.csv`.

Expected columns in this file:

- `step`
- `train_loss`
- `valid_loss`

### 3.0 Fine-Tuning Job Results & Metrics Portal Evidence

Video recording showing how to extract and review metrics from Azure AI Foundry portal:

<video src="assets/part3_fine_tuning_job_results_and_metrics.mp4" controls width="900"></video>

This video demonstrates:
- Accessing the fine-tuning job details in Foundry portal
- Viewing training progress and loss trends in real-time
- Exporting metrics as CSV for detailed analysis
- Interpreting key indicators (convergence, validation performance, overfitting signals)


In [ ]:
# ============================================================================
# METRICS ANALYSIS: Load Foundry results CSV and visualize training progress
# ============================================================================

# Path to Foundry-exported metrics CSV (should have: step, train_loss, valid_loss columns)
metrics_path = Path('outputs/results.csv')

if not metrics_path.exists():
    print('Create outputs/results.csv first (step,train_loss,valid_loss).')
else:
    # Load raw metrics from Foundry export
    raw_df = pd.read_csv(metrics_path)

    # Define required column names expected from Foundry export
    required_cols = {'step', 'train_loss', 'valid_loss'}
    missing_cols = required_cols - set(raw_df.columns)

    if missing_cols:
        print(
            f"results.csv is missing required columns: {', '.join(sorted(missing_cols))}")
    else:
        # Extract and rename columns for assignment consistency
        # step → epoch, train_loss → training_loss, valid_loss → validation_loss
        metrics_df = raw_df[['step', 'train_loss', 'valid_loss']].copy()
        metrics_df = metrics_df.rename(
            columns={
                'step': 'epoch',
                'train_loss': 'training_loss',
                'valid_loss': 'validation_loss'
            }
        )

        # Display first 50 rows of normalized metrics
        display(metrics_df.head(50))

        # ================================================================
        # Create interactive loss chart using Plotly for better exploration
        # ================================================================
        import plotly.graph_objects as go

        # Initialize figure with dual y-axis capability (if needed)
        fig = go.Figure()

        # Add training loss trace (continuous line with markers)
        fig.add_trace(
            go.Scatter(
                x=metrics_df['epoch'],
                y=metrics_df['training_loss'],
                mode='lines+markers',
                name='training_loss',
                line=dict(width=2),
                marker=dict(size=5),
                # Hover template shows epoch and loss value with 4 decimal precision
                hovertemplate='Step/Epoch: %{x}<br>Training Loss: %{y:.4f}<extra></extra>'
            )
        )

        # Add validation loss trace (separate series with square markers)
        # Filter out NaN values to avoid broken lines
        valid_points = metrics_df.dropna(subset=['validation_loss'])
        fig.add_trace(
            go.Scatter(
                x=valid_points['epoch'],
                y=valid_points['validation_loss'],
                mode='lines+markers',
                name='validation_loss',
                line=dict(width=2),
                # Square markers for distinction
                marker=dict(size=6, symbol='square'),
                hovertemplate='Step/Epoch: %{x}<br>Validation Loss: %{y:.4f}<extra></extra>'
            )
        )

        # Configure layout: titles, legend, theme
        fig.update_layout(
            title='Fine-Tuning Loss by Step/Epoch (Interactive)',
            xaxis_title='Step / Epoch',
            yaxis_title='Loss',
            template='plotly_white',  # Clean white background
            hovermode='x unified',    # Show both traces on hover
            legend=dict(orientation='h', yanchor='bottom', y=1.02,
                        xanchor='right', x=1)  # Horizontal legend above plot
        )

        # Display interactive chart (supports zoom, pan, export to PNG)
        fig.show()

        # Save normalized metrics to CSV for external analysis or reproducibility
        metrics_df.to_csv('outputs/metrics_normalized.csv', index=False)
        print('Saved normalized metrics to outputs/metrics_normalized.csv')

,epoch,training_loss,validation_loss
0,1,4.138603,NaN
1,2,4.240207,NaN
2,3,4.199733,NaN
3,4,4.397896,NaN
4,5,3.995321,NaN
5,6,4.273762,NaN
6,7,4.243929,NaN
7,8,4.116457,NaN
8,9,3.905165,NaN
9,10,3.976905,3.968811


Saved normalized metrics to metrics_normalized.csv


### 3.1 Interpretation

Based on `metrics_normalized.csv`, the model shows a strong learning trend with no major overfitting signal.

- **Did training_loss decrease consistently?** Mostly yes. It starts around ~4.14 and drops to low values near ~0.10-0.30 by later steps, with normal mini-batch fluctuations.
- **Did validation_loss follow a similar trend?** Yes. Validation checkpoints drop clearly over time (approximately from ~3.97 at early checkpoint to ~0.21 near the end), indicating better generalization.
- **Is there evidence of overfitting?** Not strong evidence. There are small temporary validation bumps (for example around some later checkpoints), but the global validation trend continues downward.
- **What would I change in the next run?**
  1. Add more challenging out-of-distribution examples in validation.
  2. Evaluate validation more frequently than every large interval to detect instability earlier.
  3. Consider early stopping around the point where validation improvements become marginal, to save cost while preserving quality.

---

## 4) Comparative Testing (Base vs Fine-Tuned)

Run at least these categories:
1. Dataset-like prompts
2. Out-of-dataset prompts
3. Edge cases
4. Direct same-prompt comparison

In [ ]:
# ============================================================================
# COMPARATIVE TESTING: Execute test prompts on both base & fine-tuned models
# ============================================================================

def ask_model(deployment_name: str, prompt: str, system_prompt: str = 'You are a helpful assistant.') -> str:
    """
    Query a model deployment with a given prompt.

    Args:
        deployment_name: Azure deployment identifier (e.g., 'gpt-4o-mini-base')
        prompt: User question to send to the model
        system_prompt: Optional system context (default: helpful assistant)

    Returns:
        response str: Model's text response (up to 400 tokens)

    Raises:
        Exception: If API call fails (e.g., invalid deployment name, quota exceeded)
    """
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.2,  # Low temperature for consistency
        max_tokens=400,   # Limit response length
    )
    return response.choices[0].message.content


# Define test prompt suite covering 4 evaluation categories
test_prompts = [
    # Category 1: Dataset-like prompts
    # These closely align with the training distribution (Azure Foundry guidance)
    {'category': 'dataset_like', 'prompt': 'Give me a 6-step checklist to launch a fine-tuning job in Azure AI Foundry, including one verification step and one common mistake to avoid.'},
    {'category': 'dataset_like', 'prompt': 'I am a beginner. Explain how to prepare training_set.jsonl and validation_set.jsonl in simple terms and include exact file names.'},
    {'category': 'dataset_like', 'prompt': 'How should I justify choosing Developer training type for an academic assignment? Keep it short and practical.'},
    {'category': 'dataset_like', 'prompt': 'Create a fast runbook to deploy a fine-tuned model and test it against the base model in the same notebook.'},

    # Category 2: Out-of-dataset prompts
    # These test generalization to nearby but broader tasks beyond training scope
    {'category': 'out_of_dataset',
        'prompt': 'Design a troubleshooting flow for when a Foundry fine-tuning job stays queued for too long.'},
    {'category': 'out_of_dataset',
        'prompt': 'What metrics besides loss would you track to evaluate assistant quality after deployment?'},
    {'category': 'out_of_dataset',
        'prompt': 'Propose a lightweight rubric to score response quality for support-chatbot prompts.'},
    {'category': 'out_of_dataset',
        'prompt': 'How would you adapt this workflow if the team needs bilingual support in English and Spanish?'},

    # Category 3: Edge cases
    # Constrained or unusual prompts requiring strict format compliance
    {'category': 'edge_case', 'prompt': 'Answer in exactly 3 bullets: my model output is empty for all prompts after deployment. What should I check first?'},
    {'category': 'edge_case', 'prompt': 'I can only run one more training due to budget. Give the single highest-impact dataset improvement and why.'},

    # Category 4: Direct comparison
    # Same prompts to both models for side-by-side quality assessment
    {'category': 'direct_comparison',
        'prompt': 'Provide a concise, numbered procedure to extract training and validation loss from Foundry and interpret overfitting risk.'},
    {'category': 'direct_comparison', 'prompt': 'Write a beginner-friendly response for: "I am lost in Azure AI Foundry menus. What exact sequence should I follow for fine-tuning and evaluation?"'}
]

# Validate that both deployment names are configured
if not BASE_DEPLOYMENT_NAME or not FINETUNED_DEPLOYMENT_NAME:
    raise ValueError(
        'Set BASE_DEPLOYMENT_NAME and FINETUNED_DEPLOYMENT_NAME in .env')

# Iterate through each test prompt and call both model deployments
rows = []
for idx, item in enumerate(test_prompts, start=1):
    p = item['prompt']
    try:
        # Query base model deployment
        base_answer = ask_model(BASE_DEPLOYMENT_NAME, p)
        # Query fine-tuned model deployment
        ft_answer = ask_model(FINETUNED_DEPLOYMENT_NAME, p)
        # No error if both calls succeeded
        error_msg = ''
    except Exception as exc:
        # Capture error details if API call failed
        base_answer = ''
        ft_answer = ''
        error_msg = f'{type(exc).__name__}: {exc}'

    # Append result row with prompt metadata and outputs
    rows.append({
        'prompt_id': idx,
        'category': item['category'],
        'prompt': p,
        'base_output': base_answer,
        'finetuned_output': ft_answer,
        'qualitative_judgment': '',  # User fills this after manual review
        'error': error_msg,
    })

# Create DataFrame from results and export to CSV for grading/analysis
comparison_df = pd.DataFrame(rows)
display(comparison_df)
comparison_df.to_csv('outputs/comparison_results.csv', index=False)
print('Saved: outputs/comparison_results.csv')

,prompt_id,category,prompt,base_output,finetuned_output,qualitative_judgment,error
0,1,dataset_like,Give me a 6-step checklist to launch a fine-tu...,Certainly! Here’s a 6-step checklist to launch...,Here’s a concise 6-step checklist for launchin...,,
1,2,dataset_like,I am a beginner. Explain how to prepare traini...,Certainly! Preparing `training_set.jsonl` and ...,Here’s a beginner-friendly approach to prepari...,,
2,3,dataset_like,How should I justify choosing Developer traini...,When justifying the choice of Developer traini...,Justification for choosing Developer training ...,,
3,4,dataset_like,Create a fast runbook to deploy a fine-tuned m...,Certainly! Below is a fast runbook for deployi...,Here’s a concise and efficient runbook for dep...,,
4,5,out_of_dataset,Design a troubleshooting flow for when a Found...,When a Foundry fine-tuning job stays queued fo...,Here's a structured troubleshooting flow for w...,,
5,6,out_of_dataset,What metrics besides loss would you track to e...,Evaluating the quality of an AI assistant afte...,When evaluating assistant quality after deploy...,,
6,7,out_of_dataset,Propose a lightweight rubric to score response...,Certainly! Here’s a lightweight rubric to eval...,Here’s a lightweight rubric to score response ...,,
7,8,out_of_dataset,How would you adapt this workflow if the team ...,To adapt a workflow for bilingual support in E...,To adapt a workflow for bilingual support in E...,,
8,9,edge_case,Answer in exactly 3 bullets: my model output i...,- **Model Configuration**: Verify that the mod...,- Verify that the deployment configuration mat...,,
9,10,edge_case,I can only run one more training due to budget...,To maximize the impact of a single training ru...,The single highest-impact dataset improvement ...,,


Saved: comparison_results.csv


### 4.1 Qualitative Analysis

- **Accuracy/Faithfulness**: The fine-tuned model was usually more aligned with the assignment context (Foundry workflow, practical steps, concise guidance). However, it still produced occasional factual drift in specific details.
- **Style Consistency**: The fine-tuned model was consistently more procedural and concise than the base model, which tended to be more verbose and generic.
- **Format Compliance**: The fine-tuned model generally followed requested structure better (checklists, short runbooks, constrained style), especially for prompts 1, 4, 9, and 12.
- **Edge-case Handling**: On constrained prompts, the fine-tuned model stayed concise and instruction-following (notably prompt 9), but quality varied on highest-impact recommendation quality (prompt 10).

Detailed outcome by prompt group:

1. **Prompts where fine-tuned clearly outperformed base**: `1, 4, 9, 12`
2. **Prompts where outputs were similar quality**: `3, 5, 6, 7, 8`
3. **Prompts where fine-tuned underperformed or was less precise**: `2, 10, 11`

Main conclusions:

4. **Main improvement observed**: The fine-tuned model improved response style control (clear steps, concise tone, assignment-oriented guidance) and reduced unnecessary verbosity.
5. **Main remaining weakness**: Some answers still include partially off-target content or weak specificity (for example exact file naming/strict task focus in prompts 2, 10, and 11).
6. **Next dataset improvement for retraining**: Add targeted corrective examples for strict instruction-following and factual precision, especially for prompts requiring exact file names and single high-impact recommendations.